# logit-thermo A2-6：分析（乳の種類、CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 15〜30 分。**データ集めの 3 本が全部終わってから**回してください。
- `A2_5/` の Pythia 6 回と `A2_6/` の 9 回を読み、ゲートの判定を計算して `A2_6/a2_6_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：`scripts/a2_2_analyze.py`・`scripts/a2_5_analyze.py` は Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_6_analyze.py`（`283e07c6ee1bc3af…`）は埋め込み（データを開く前に固定）。記事の本文は使いません。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-6 分析（乳の種類、判定と要約、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-6 分析（乳の種類、判定と要約、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-6 analyze', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCA22': 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7', 'SRC5A': 'f4ef35b9f594a205ac1145c1a326e9de8a25280171fe2b93083cc1676e58d3ae'})
SRCA22, SRC5A = _s['SRCA22'], _s['SRC5A']
SRC6A = '"""A2-6（ゲート docs/A2_6_GATE.md）の分析：「乳の種類」テスト。データを開く前に固定する。CPU。\nscripts/a2_2_analyze.py（per_doc_feedback・feedback_stats・crossing_mono・pav）と scripts/a2_5_analyze.py（spearman）を\n実行したのと同じ名前空間で実行する。記事の本文は使わない（npz の tf と gen だけ）。\n\n    analyze6(a2_5 のフォルダ, a2_6 のフォルダ) -> dict\n"""\nimport math\nimport numpy as np\n\nA6_FAMS = {  # 家族 → [(名前, 名目のパラメータの数)]。Pythia の 6 回は A2-5 のデータ（同じ記事・同じ生成の種）\n    \'Pythia\': [(\'pythia-70m\', 70e6), (\'pythia-160m\', 160e6), (\'pythia-410m\', 410e6), (\'pythia-1b\', 1.0e9),\n               (\'pythia-1.4b\', 1.4e9), (\'pythia-2.8b\', 2.8e9)],\n    \'Qwen2.5\': [(\'qwen2.5-0.5b\', 0.5e9), (\'qwen2.5-1.5b\', 1.5e9), (\'qwen2.5-3b\', 3.0e9)],\n    \'SmolLM2\': [(\'smollm2-135m\', 135e6), (\'smollm2-360m\', 360e6), (\'smollm2-1.7b\', 1.7e9)],\n    \'Qwen3\': [(\'qwen3-0.6b\', 0.6e9), (\'qwen3-1.7b\', 1.7e9), (\'qwen3-4b\', 4.0e9)],\n}\nA6_NEW = [\'Qwen2.5\', \'SmolLM2\', \'Qwen3\']          # Q9・Q10 で数える家族（Pythia は A2-5 で見たので数えない）\nA6_BOOT, A6_BOOT_SEED = 1000, 20261103\nA6_PERM, A6_PERM_SEED = 20000, 20261104\nA6_ETA_LINE = 0.5\nA6_T07 = 0.7\nA6_REL = 0.83                                     # Q10 の二つめの温度：T_0 × 0.83（温度 0.7 ÷ A2-5 の T_0 のだいたい）\nA6_EQ_PT, A6_EQ_BOOT = 1.05, 1.08                 # Q9「動かない」：家族の中の T_0 の最大 ÷ 最小（点の値・ブートストラップの 95% 点）\n\n\ndef eta2(y, labels):\n    """全体のばらつきのうち、家族の平均の違いで説明できる割合（η²）。"""\n    y = np.asarray(y, float); labels = np.asarray(labels)\n    tot = ((y - y.mean()) ** 2).sum()\n    if tot == 0:\n        return float(\'nan\')\n    btw = sum((labels == g).sum() * (y[labels == g].mean() - y.mean()) ** 2 for g in np.unique(labels))\n    return float(btw / tot)\n\n\ndef perm_p(y, labels, n, seed):\n    rng = np.random.default_rng(seed)\n    obs = eta2(y, labels)\n    lab = np.asarray(labels)\n    hit = sum(eta2(y, rng.permutation(lab)) >= obs - 1e-12 for _ in range(n))\n    return float((1 + hit) / (1 + n))\n\n\ndef fam_dir(xs, yp, yb, pos, neg):\n    """3 点（以上）の家族の向き：ρ が ±1 ちょうどで、ブートストラップの 95% 以上が同じ符号のときだけ向きを言う。"""\n    r = spearman(xs, yp)\n    ok = np.isfinite(yb).all(1)\n    rb = np.array([spearman(xs, b) for b in yb[ok]]) if ok.any() else np.array([])\n    fp = float((rb > 0).sum() / len(yb)); fn = float((rb < 0).sum() / len(yb))\n    w = pos if (r >= 1 - 1e-9 and fp >= 0.95) else neg if (r <= -1 + 1e-9 and fn >= 0.95) else \'はっきりしない\'\n    return w, dict(rho=r, frac_pos=fp, frac_neg=fn, boot_ok=float(ok.mean()))\n\n\ndef q8_word(names, T0pt, T0b, fam_of):\n    """log T_0 の η²・並べ替えの p・ブートストラップの区間から Q8 の言葉を出す。"""\n    T0 = np.array([T0pt[n] for n in names]); lab = [fam_of[n] for n in names]\n    if not np.isfinite(T0).all():\n        return \'はっきりしない\', dict(missing_T0=[n for n in names if not np.isfinite(T0pt[n])])\n    y = np.log(T0)\n    e = eta2(y, lab); p = perm_p(y, lab, A6_PERM, A6_PERM_SEED)\n    Yb = np.log(np.stack([T0b[n] for n in names], 1))\n    ok = np.isfinite(Yb).all(1)\n    eb = np.array([eta2(Yb[b], lab) for b in np.nonzero(ok)[0]])\n    lo, hi = (float(np.percentile(eb, 2.5)), float(np.percentile(eb, 97.5))) if ok.mean() >= 0.8 else (float(\'nan\'), float(\'nan\'))\n    if ok.mean() < 0.8:\n        w = \'はっきりしない\'\n    elif p <= 0.05 and lo >= A6_ETA_LINE:\n        w = \'家族で決まる\'\n    elif p > 0.2 and hi < A6_ETA_LINE:\n        w = \'家族の効き目は見えない\'\n    else:\n        w = \'はっきりしない\'\n    return w, dict(eta2=e, perm_p=p, boot_ci=[lo, hi], boot_ok=float(ok.mean()),\n                   chance=(len({fam_of[n] for n in names}) - 1) / (len(names) - 1))\n\n\ndef F_rel(F, temps, T0):\n    """T_0 × A6_REL の温度での F（対数温度で直線補間）。"""\n    if not np.isfinite(T0):\n        return float(\'nan\')\n    Ts = A6_REL * T0\n    if not (temps[0] <= Ts <= temps[-1]):\n        return float(\'nan\')\n    return float(np.interp(math.log(Ts), np.log(temps), F))\n\n\ndef analyze6_from_fd(fds, temps):\n    """fds：{名前: (nD, nT)}。全モデルで記事の並びが同じであること（共通の記事にそろえてから渡す）。"""\n    nD = {v.shape[0] for v in fds.values()}\n    assert len(nD) == 1\n    nD = nD.pop()\n    k07 = temps.index(A6_T07)\n    names = [n for f in A6_FAMS.values() for n, _ in f]\n    new9 = [n for g in A6_NEW for n, _ in A6_FAMS[g]]\n    fam_of = {n: g for g, f in A6_FAMS.items() for n, _ in f}\n    res = {\'models\': {}, \'n_docs\': nD}\n    eps = {}\n    for n in names:\n        assert np.isfinite(fds[n]).all(), (\'f に有限でない値があります\', n)\n        fs = feedback_stats(fds[n], temps, np.random.default_rng(0))\n        eps[n] = fs[\'eps\']\n        F = np.median(fds[n], 0)\n        res[\'models\'][n] = dict(fam=fam_of[n], T0=fs[\'T0\'], T0_ci=fs[\'T0_ci\'], eps=fs[\'eps\'], F07=float(F[k07]),\n                                Frel=F_rel(F, temps, fs[\'T0\']),\n                                s=float((F[temps.index(0.9)] - F[temps.index(0.8)]) / 0.1), F=F.tolist())\n    eps_c = float(np.median(list(eps.values())))\n    for n in names:\n        res[\'models\'][n][\'T0_common_eps\'] = crossing_mono(np.median(fds[n], 0), temps, eps_c)\n    rng = np.random.default_rng(A6_BOOT_SEED)\n    idx = rng.integers(0, nD, size=(A6_BOOT, nD))\n    Fb = {n: np.stack([np.median(fds[n][idx[b]], 0) for b in range(A6_BOOT)]) for n in names}   # (B, nT)\n    T0b = {n: np.array([crossing_mono(Fb[n][b], temps, eps[n]) for b in range(A6_BOOT)]) for n in names}\n    T0bc = {n: np.array([crossing_mono(Fb[n][b], temps, eps_c) for b in range(A6_BOOT)]) for n in names}\n    F07b = {n: Fb[n][:, k07] for n in names}\n    Frelb = {n: np.array([F_rel(Fb[n][b], temps, T0b[n][b]) for b in range(A6_BOOT)]) for n in names}\n    # Q8：T_0 は家族で決まるか。3 つの見方（15 モデル・新しい 9 モデル・共通の ε）がそろったときだけ言う\n    T0pt = {n: res[\'models\'][n][\'T0\'] for n in names}\n    T0ptc = {n: res[\'models\'][n][\'T0_common_eps\'] for n in names}\n    views = {\'all15\': q8_word(names, T0pt, T0b, fam_of), \'new9\': q8_word(new9, T0pt, T0b, fam_of),\n             \'all15_common_eps\': q8_word(names, T0ptc, T0bc, fam_of)}\n    ws = {v[0] for v in views.values()}\n    res[\'Q8\'] = ws.pop() if len(ws) == 1 else \'はっきりしない\'\n    res[\'Q8_detail\'] = {k: dict(word=v[0], **v[1]) for k, v in views.items()}\n    res[\'eps_common\'] = eps_c\n    # Q9：家族の中で T_0 が大きさで動くか（動かない家族の数）／ Q10：大きいほど浅く固まる家族の数\n    res[\'Q9_by_fam\'], res[\'Q10_by_fam\'] = {}, {}\n    for g, mem in A6_FAMS.items():\n        xs = [math.log(p) for _, p in mem]; ns_ = [n for n, _ in mem]\n        yt = np.array([T0pt[n] for n in ns_])\n        Yb = np.stack([T0b[n] for n in ns_], 1)\n        if np.isfinite(yt).all():\n            w, d = fam_dir(xs, np.log(yt), np.log(Yb), \'大きいほど高い\', \'大きいほど低い\')\n            ratio = float(yt.max() / yt.min())\n            okb = np.isfinite(Yb).all(1)\n            rb = Yb[okb].max(1) / Yb[okb].min(1)\n            r95 = float(np.percentile(rb, 95)) if okb.mean() >= 0.8 else float(\'nan\')\n            if w == \'はっきりしない\' and ratio <= A6_EQ_PT and r95 <= A6_EQ_BOOT:\n                w = \'動かない\'\n            d.update(ratio=ratio, ratio_boot95=r95)\n        else:\n            w, d = \'はっきりしない\', dict(missing=True)\n        res[\'Q9_by_fam\'][g] = dict(word=w, **d)\n        w1, d1 = fam_dir(xs, [res[\'models\'][n][\'F07\'] for n in ns_], np.stack([F07b[n] for n in ns_], 1), \'大きいほど浅い\', \'大きいほど深い\')\n        w2, d2 = fam_dir(xs, [res[\'models\'][n][\'Frel\'] for n in ns_], np.stack([Frelb[n] for n in ns_], 1), \'大きいほど浅い\', \'大きいほど深い\')\n        res[\'Q10_by_fam\'][g] = dict(word=w1 if w1 == w2 else \'はっきりしない\', F07=dict(word=w1, **d1), Frel=dict(word=w2, **d2))\n    res[\'Q9\'] = sum(res[\'Q9_by_fam\'][g][\'word\'] == \'動かない\' for g in A6_NEW)\n    res[\'Q9_counts\'] = {w: sum(res[\'Q9_by_fam\'][g][\'word\'] == w for g in A6_NEW) for w in (\'動かない\', \'大きいほど高い\', \'大きいほど低い\', \'はっきりしない\')}\n    res[\'Q10\'] = sum(res[\'Q10_by_fam\'][g][\'word\'] == \'大きいほど浅い\' for g in A6_NEW)\n    # 記録：家族ごとの T_0 の平均（対数で平均して戻す）と幅\n    res[\'fam_T0\'] = {g: dict(geo_mean=float(np.exp(np.mean(np.log([T0pt[n] for n, _ in mem])))),\n                             range=[float(min(T0pt[n] for n, _ in mem)), float(max(T0pt[n] for n, _ in mem))])\n                     for g, mem in A6_FAMS.items() if all(np.isfinite(T0pt[n]) for n, _ in mem)}\n    return res\n\n\ndef analyze6(d5, d6):\n    fams_dir = {n: (d5 if g == \'Pythia\' else d6) for g, f in A6_FAMS.items() for n, _ in f}\n    raw, temps, uses = {}, None, {}\n    for n, d in fams_dir.items():\n        Z = np.load(f\'{d}/{n}.npz\')\n        t = [float(x) for x in Z[\'temps\']]\n        assert temps is None or t == temps, \'温度の並びが違います\'\n        temps = t\n        uses[n] = [int(u) for u in Z[\'use\']]\n        raw[n] = per_doc_feedback(Z[\'tf\'][0], Z[\'gen\'][0])\n    common = sorted(set.intersection(*[set(u) for u in uses.values()]))\n    fds = {n: raw[n][[uses[n].index(i) for i in common]] for n in raw}\n    r = analyze6_from_fd(fds, temps)\n    r[\'common_docs\'] = common\n    r[\'dropped_docs\'] = {n: sorted(set(u) - set(common)) for n, u in uses.items() if set(u) - set(common)}\n    return r\n\n\ndef self_test_a26_analyze():\n    assert abs(eta2([0, 0, 1, 1], [\'a\', \'a\', \'b\', \'b\']) - 1) < 1e-12\n    assert abs(eta2([0, 1, 0, 1], [\'a\', \'a\', \'b\', \'b\'])) < 1e-12\n    temps = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05, 1.1]\n    rng = np.random.default_rng(3)\n    T0fam = {\'Pythia\': 0.84, \'Qwen2.5\': 0.91, \'SmolLM2\': 0.82, \'Qwen3\': 0.87}   # 交点の両どなりに温度 0.7 が入らないように\n    fds = {}\n    for g, mem in A6_FAMS.items():\n        for j, (n, _) in enumerate(mem):\n            depth = 3.0 - 0.6 * j                         # 大きいほど浅い（F07 が上がる）\n            F = np.array([(t - T0fam[g]) * (8 if t < T0fam[g] else 4) for t in temps])\n            F[1] = -depth if temps[1] < T0fam[g] else F[1]\n            fds[n] = F[None, :] + rng.normal(0, 0.05, (40, len(temps)))\n    r = analyze6_from_fd(fds, temps)\n    assert r[\'Q8\'] == \'家族で決まる\', r[\'Q8_detail\']\n    assert r[\'Q10\'] == 3, r[\'Q10_by_fam\']\n    assert r[\'Q9\'] == 3, r[\'Q9_by_fam\']                # 家族の中で T_0 は同じ → 3 家族とも「動かない」\n    assert r[\'Q8_detail\'][\'new9\'][\'word\'] == \'家族で決まる\', r[\'Q8_detail\']\n    # 家族の効き目がないとき\n    fds2 = {n: (np.array([(t - 0.85) * 6 for t in temps])[None, :] + rng.normal(0, 0.05, (40, len(temps)))) for n in fds}\n    r2 = analyze6_from_fd(fds2, temps)\n    assert r2[\'Q8\'] in (\'家族の効き目は見えない\', \'はっきりしない\'), r2[\'Q8_detail\']\n    # 雑音が大きく、家族の効き目がないとき：「家族で決まる」は出ない\n    fds3 = {n: (np.array([(t - 0.85) * 6 for t in temps])[None, :] + rng.normal(0, 0.6, (40, len(temps)))) for n in fds}\n    r3 = analyze6_from_fd(fds3, temps)\n    assert r3[\'Q8\'] != \'家族で決まる\', r3[\'Q8_detail\']\n    # 有限でない値は黙って通さない\n    bad = dict(fds); bad[\'qwen3-4b\'] = bad[\'qwen3-4b\'].copy(); bad[\'qwen3-4b\'][0, 0] = np.nan\n    try:\n        analyze6_from_fd(bad, temps); raise RuntimeError(\'通ってしまった\')\n    except AssertionError:\n        pass\n    return \'OK\'\n'
assert hashlib.sha256(SRC6A.encode()).hexdigest() == '283e07c6ee1bc3afaf0f6e30ab06bd07c034a00a30b2ac8f1815871074d91cd4'
exec(SRCA22)
exec(SRC5A)
exec(SRC6A)
print('自己テスト：', self_test_a26_analyze())


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
try:
    r = analyze6('/content/drive/MyDrive/logit-thermo/A2_5', '/content/drive/MyDrive/logit-thermo/A2_6')
    r['a2_6_analyze_sha256'] = '283e07c6ee1bc3afaf0f6e30ab06bd07c034a00a30b2ac8f1815871074d91cd4'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_6' + '/a2_6_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-6 の分析で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定と要約を A2_6/a2_6_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
